# LOSO Binary Total-Cloud and Water Accuracy

This notebook evaluates the already-fitted leave-one-site-out DT, RF, and XGBoost models. It does **not** tune or fit any model. Original classes 1 (thin cloud) and 2 (cloud affected) are merged into **total cloud**; original class 3 remains **water**.

Class-specific `total_cloud_accuracy` and `water_accuracy` are producer accuracies (recalls): the fraction of actual rows in each binary class that are classified correctly. `overall_binary_accuracy` is the fraction of all binary predictions that are correct, and `balanced_binary_accuracy` is the mean of the two class-specific accuracies.

In [ ]:
import importlib
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import lswt_cloud_masking.loso_binary_evaluation as binary_eval
importlib.reload(binary_eval)
from lswt_cloud_masking.loso_binary_evaluation import evaluate_saved_loso_binary_models

## 1. Locate the completed LOSO experiment

The evaluator reads the saved datasets, feature metadata, and final model files under this directory. It writes only new metric summaries under `binary_evaluation/`.

In [ ]:
LOSO_OUTPUT_DIR = ROOT / "outputs" / "leave_one_site_out"
BINARY_OUTPUT_DIR = LOSO_OUTPUT_DIR / "binary_evaluation"

print("Saved LOSO artifacts:", LOSO_OUTPUT_DIR)
print("Binary metric outputs:", BINARY_OUTPUT_DIR)

## 2. Load the final models and calculate binary metrics

This step performs prediction only. It evaluates 6 held-out sites × 3 saved models × 3 datasets = 54 combinations. Large Random Forest files are loaded one at a time to limit memory use.

In [ ]:
binary_result = evaluate_saved_loso_binary_models(
    LOSO_OUTPUT_DIR,
    output_dir=BINARY_OUTPUT_DIR,
)
binary_result["paths"]

## 3. Mean robustness across the six held-out sites

In [ ]:
summary_columns = [
    "model",
    "dataset",
    "overall_binary_accuracy_mean",
    "balanced_binary_accuracy_mean",
    "total_cloud_accuracy_mean",
    "water_accuracy_mean",
    "total_cloud_accuracy_std",
    "water_accuracy_std",
]
display(
    binary_result["summary"][summary_columns]
    .sort_values(["dataset", "model"], kind="stable")
    .style.format({column: "{:.3f}" for column in summary_columns[2:]})
)

## 4. Site-level cloud and water accuracies

In [ ]:
detailed = binary_result["detailed"]
for dataset_name in ("train", "test", "loso"):
    print(f"\n{dataset_name.upper()}")
    table = detailed.loc[detailed["dataset"].eq(dataset_name)].pivot(
        index="held_out_site",
        columns="model",
        values=["total_cloud_accuracy", "water_accuracy", "overall_binary_accuracy"],
    )
    display(table.style.format("{:.3f}"))

In [ ]:
plot_data = detailed.melt(
    id_vars=["held_out_site", "model", "dataset"],
    value_vars=["total_cloud_accuracy", "water_accuracy"],
    var_name="binary_class",
    value_name="accuracy",
)
fig, axes = plt.subplots(3, 2, figsize=(16, 13), sharey=True)
for row_index, dataset_name in enumerate(("train", "test", "loso")):
    for column_index, metric_name in enumerate(("total_cloud_accuracy", "water_accuracy")):
        axis = axes[row_index, column_index]
        subset = plot_data.loc[
            plot_data["dataset"].eq(dataset_name)
            & plot_data["binary_class"].eq(metric_name)
        ]
        sns.barplot(
            data=subset,
            x="held_out_site",
            y="accuracy",
            hue="model",
            ax=axis,
        )
        axis.set_ylim(0, 1)
        axis.set_title(f"{dataset_name.upper()}: {metric_name.replace('_', ' ')}")
        axis.tick_params(axis="x", rotation=30)
        if row_index != 0 or column_index != 1:
            axis.get_legend().remove()
axes[0, 1].legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()

## 5. Audit the binary confusion counts

These four columns contain the raw counts behind every reported accuracy.

In [ ]:
count_columns = [
    "held_out_site", "model", "dataset", "n_evaluated_rows",
    "true_total_cloud_predicted_total_cloud",
    "true_total_cloud_predicted_water",
    "true_water_predicted_total_cloud",
    "true_water_predicted_water",
]
detailed[count_columns]

The detailed, across-site summary, wide comparison, and metric-definition files are saved in `outputs/leave_one_site_out/binary_evaluation/`. Re-running this notebook overwrites only those derived evaluation files; it does not modify the trained models or original LOSO datasets.